In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

!pip install -q torch torchvision torchaudio transformers peft bitsandbytes accelerate datasets
# If you have a requirements.txt, you can upload it and run: !pip install -q -r /content/drive/MyDrive/OCR_Project/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00


In [3]:
import os
os.chdir('/content/drive/MyDrive/Project2_OCR')
!ls

data	 evaluate.py	main.py       outputs	  qwen_env	    train.py
eval.py  evaluation.py	OCR_Training  preprocess  requirements.txt


In [4]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 50.5 MB/s eta 0:00:00


In [5]:
!pip install -q -r /content/drive/MyDrive/Project2_OCR/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.5 MB/s eta 0:00:00


In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [7]:
!pip install -q huggingface_hub

In [8]:
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="abdoelsayed/CORU",
    repo_type="dataset"
)

print(dataset_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--abdoelsayed--CORU/snapshots/c3c4b97b232bbe03046c78a974f516d439c1124e


In [9]:
import zipfile
import os

root = dataset_path

zip_path = os.path.join(root, "Receipt", "train.zip")
extract_dir = "/content/CORU/train"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extracted!")

Extracted!


In [10]:
for split in ["train", "val", "test"]:
    zip_path = os.path.join(root, "Receipt", f"{split}.zip")
    extract_dir = f"/content/CORU/{split}"

    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

### CORUDataset Loader

This custom PyTorch `Dataset` class will load images and their associated annotations (bounding boxes and labels) from the CORU dataset. It reads image files and parses the COCO-style annotations from the provided JSON files.

### Demonstration of CORUDataset

In [16]:
!python train.py \
    --dataset_name "/content/CORU" \
    --dataset_split "train" \
    --output_dir "./outputs/coru_run" \
    --max_samples 200 \
    --per_device_train_batch_size 4

CUDA: available — device 0 = Tesla T4 (capability (7, 5))
Fetching 2 files: 100% 2/2 [00:00<00:00, 14563.56it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 729/729 [00:00<00:00, 5503.65it/s]
Creating new LoRA adapter...
trainable params: 22,436,224 || all params: 2,231,421,824 || trainable%: 1.0055
Model moved to GPU: cuda:0
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset '/content/CORU' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Resolving data files: 100% 14062/14062 [00:00<00:00, 16336.50it/s]
Resolving data files: 100% 1600/1600 [00:00<00:00, 30735.10it/s]
Resolving data files: 100% 5314/5314 [00:00<00:00, 16173.91it/s]
Generating train split: 6024 examples [00:00, 23974.87 examples/s]
Generating validation split: 800 examples [00:00, 31299.02 examples/s]
Generating 

In [22]:
import json

with open("/content/CORU/repo/Receipt/train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data.keys())

for k, v in data.items():
    print("\nKEY:", k)
    print(type(v))
    if isinstance(v, list):
        print("Length:", len(v))
        print("First item:")
        print(v[0])

dict_keys(['images', 'annotations', 'categories'])

KEY: images
<class 'list'>
Length: 8038
First item:
{'file_name': '0009385a-6e1d-408a-a346-2f50bfd35f71.jpg', 'id': 0, 'width': 3472, 'height': 4624}

KEY: annotations
<class 'list'>
Length: 57028
First item:
{'image_id': 0, 'bbox': [822, 2322, 1754, 95], 'category_id': 4, 'id': 0, 'iscrowd': 0, 'area': 166630, 'segmentation': [[822, 2322, 2576, 2322, 2576, 2417, 822, 2417]]}

KEY: categories
<class 'list'>
Length: 5
First item:
{'id': 0, 'name': '0'}


In [11]:
import shutil
import os

target = "/content/CORU/repo"

if os.path.exists(target):
    shutil.rmtree(target)

shutil.copytree(dataset_path, target)

print(target)

/content/CORU/repo


In [12]:
import zipfile
import os

# The base path where the entire CORU dataset was copied (by YRBo2daUajJP)
repo_base_path = "/content/CORU/repo"

# Iterate over all splits to extract OCR data
for split in ["train", "val", "test"]:
    # Construct the path to the OCR zip file within the copied repository
    zip_path = os.path.join(repo_base_path, "OCR", f"{split}.zip")
    # Construct the target extraction directory for the OCR data
    extract_dir = f"/content/CORU/OCR/{split}"

    # Create the extraction directory if it doesn't exist
    os.makedirs(extract_dir, exist_ok=True)

    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        print(f"Extracted {zip_path} to {extract_dir}")
    else:
        print(f"Zip file not found: {zip_path}")

# --- BEGIN NEW DEBUGGING SECTION ---
# Verify extraction more deeply for the 'train' split
print(f"\nVerifying contents of /content/CORU/OCR/train/train/:")
nested_ocr_train_dir = "/content/CORU/OCR/train/train/"
if os.path.exists(nested_ocr_train_dir):
    for item in os.listdir(nested_ocr_train_dir):
        item_path = os.path.join(nested_ocr_train_dir, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")
    if len(os.listdir(nested_ocr_train_dir)) > 10:
        print("  ... (more files/dirs)")
else:
    print(f"Directory {nested_ocr_train_dir} does not exist.")
# --- END NEW DEBUGGING SECTION ---

Streaming output truncated to the last 5000 lines.
  [FILE] 908e863a-cff5-4653-ad57-73769ca3a147_line_7.jpg
  [FILE] aafd886d-0b80-4750-9fdf-89e06f669ee1.txt
  [FILE] b3a860ba-fc50-4b3c-8619-c1a3559e227a_line_2.txt
  [FILE] 198559b3-df57-4ec6-993b-a8c71a18f275_line_11.txt
  [FILE] 0c62c65c-884d-4237-8eb5-4ec376b2ccd1_line_11.txt
  [FILE] 0d80baf1-14f7-44cd-b539-0c39622189ea_total.jpg
  [FILE] 55440a56-0379-4712-a6e0-a8b16388cb23_line_7.txt
  [FILE] 7ba01d60-05c1-4aa5-b167-7792fb833ac6_line_15.jpg
  [FILE] cab1038b-f2ea-40e6-88f9-913552c93d51_line_30.jpg
  [FILE] d041bfed-5fe8-403d-a6da-e32331ce4e13_line_6.txt
  [FILE] 05bd5e4c-a0fc-426d-b481-8ba97e61734d_line_18.jpg
  [FILE] 69a0301d-c885-4496-9c01-37026df200a0_line_24.jpg
  [FILE] 74bc9258-5d06-44fe-ad5d-147782674829_line_19.txt
  [FILE] 53dc48f6-a393-40c1-a0f0-6be3a79f9f36_line_14.txt
  [FILE] a0c3686f-fdf3-4391-9a8d-c833925db5a2_line_3.txt
  [FILE] 3f43cd41-cee1-4986-bafa-37b1b73b4f4c_line_10.jpg
  [FILE] 6ba2c67d-2652-4c3b-9c2a-58b

### OCRDataset Loader

This custom PyTorch `Dataset` class will load images and their associated text annotations for the OCR task. It reads image files and parses bounding box and text information from corresponding `.txt` files. The dataset accounts for a nested directory structure where the extracted content is under a subdirectory named after the split (e.g., `/content/CORU/OCR/train/train/`).

In [48]:
import torch
from PIL import Image
import os
import numpy as np

class OCRDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split="train"):
        self.root_dir = root_dir
        self.split = split

        # Correct paths to account for the nested 'split' directory
        self.data_dir = os.path.join(root_dir, split, split) # e.g., /content/CORU/OCR/train/train

        # Image and label files are directly within data_dir, not in subdirectories
        self.image_dir = self.data_dir # Image files are directly here
        self.label_dir = self.data_dir # Label files are directly here

        if not os.path.exists(self.image_dir):
            raise FileNotFoundError(f"Image directory not found: {self.image_dir}")
        if not os.path.exists(self.label_dir):
            raise FileNotFoundError(f"Label directory not found: {self.label_dir}")

        # Filter for image files only
        self.image_files = [f for f in os.listdir(self.image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.image_files.sort() # Ensure consistent order

    def __len__(self):
        return len(self.image_files)

    def _crop_to_content(self, image, threshold=240, margin=2):
        """
        Crops a PIL image to the bounding box of its non-background content.
        Assumes lighter background (e.g., white) and darker foreground (text).
        Returns the cropped image and its new dimensions (width, height).
        """
        # Convert to grayscale
        gray_image = image.convert('L')
        np_image = np.array(gray_image)

        # Find rows and columns that contain pixels darker than the threshold
        # (i.e., contain text/content)
        rows_with_content = np.any(np_image < threshold, axis=1)
        cols_with_content = np.any(np_image < threshold, axis=0)

        if not np.any(rows_with_content) or not np.any(cols_with_content):
            # No content found (e.g., entirely white image), return original
            return image, image.width, image.height

        y_min, y_max = np.where(rows_with_content)[0][[0, -1]]
        x_min, x_max = np.where(cols_with_content)[0][[0, -1]]

        # Apply margin, ensuring coordinates stay within image bounds
        y_min = max(0, y_min - margin)
        y_max = min(image.height - 1, y_max + margin)
        x_min = max(0, x_min - margin)
        x_max = min(image.width - 1, x_max + margin)

        # PIL's crop method uses (left, upper, right, lower) where right and lower are exclusive.
        # So, we add 1 to x_max and y_max to include the last row/column.
        cropped_image = image.crop((x_min, y_min, x_max + 1, y_max + 1))
        new_width, new_height = cropped_image.size
        return cropped_image, new_width, new_height

    def __getitem__(self, idx):
        img_filename = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_filename)
        label_filename = img_filename.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_filename)

        # Load image
        image = Image.open(img_path).convert('RGB')

        # Apply content-aware cropping to remove padding
        image, img_width, img_height = self._crop_to_content(image)

        # Load annotations
        boxes = []
        labels = []
        texts = []

        if os.path.exists(label_path):
            with open(label_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                # print(f"Content of {label_path}: {content}") # For debugging

                # Check if content starts with '["' and ends with '"]'
                if content.startswith('["') and content.endswith('"]'):
                    extracted_text = content[2:-2] # Remove [" and "]

                    # For cropped line images, the bbox is the full image
                    boxes.append([0, 0, img_width, img_height])
                    labels.append(0) # Use a generic label for text (if no specific categories are provided for OCR)
                    texts.append(extracted_text)
                else:
                    # If it's not the "[\"TEXT\"]" format, or it's empty, handle accordingly
                    print(f"Warning: Unexpected or empty label file format for {label_path}: {content}")

        # Convert to tensors
        # Ensure that if no annotations are found, empty tensors of correct shape are returned
        if not boxes:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "texts": texts, # Raw text labels
            "image_id": torch.tensor([idx]) # A simple image ID for this batch
        }

        return image, target

### Demonstration of OCRDataset

In [49]:
# Instantiate the OCR dataset for the 'train' split
ocr_train_dataset = OCRDataset(root_dir='/content/CORU/OCR', split='train')

print(f"Number of images in OCR train dataset: {len(ocr_train_dataset)}")
for i in range(1,6):
# Get a single item from the dataset (e.g., the first image)
  image_ocr, target_ocr = ocr_train_dataset[i]

  print(f"\nType of image_ocr: {type(image_ocr)}")
  print(f"Image_ocr size: {image_ocr.size}")
  print(f"Target_ocr keys: {target_ocr.keys()}")
  print(f"Bounding boxes (first 5 if any):\n{target_ocr['boxes'][:5]}")
  print(f"Labels (first 5 if any):\n{target_ocr['labels'][:5]}")
  print(f"Texts (first 5 if any):\n{target_ocr['texts'][:5]}")
  print(f"Image ID: {target_ocr['image_id']}")



Number of images in OCR train dataset: 21103

Type of image_ocr: <class 'PIL.Image.Image'>
Image_ocr size: (929, 48)
Target_ocr keys: dict_keys(['boxes', 'labels', 'texts', 'image_id'])
Bounding boxes (first 5 if any):
tensor([[  0.,   0., 929.,  48.]])
Labels (first 5 if any):
tensor([0])
Texts (first 5 if any):
['P<EGYELBENDARY<<SAMEH<BAHGAT<AHMED<<<<<<<<<<']
Image ID: tensor([1])

Type of image_ocr: <class 'PIL.Image.Image'>
Image_ocr size: (913, 42)
Target_ocr keys: dict_keys(['boxes', 'labels', 'texts', 'image_id'])
Bounding boxes (first 5 if any):
tensor([[  0.,   0., 913.,  42.]])
Labels (first 5 if any):
tensor([0])
Texts (first 5 if any):
['A195715602EGY7203132F2312123<<<<<<<<<<<<<<00']
Image ID: tensor([2])

Type of image_ocr: <class 'PIL.Image.Image'>
Image_ocr size: (929, 44)
Target_ocr keys: dict_keys(['boxes', 'labels', 'texts', 'image_id'])
Bounding boxes (first 5 if any):
tensor([[  0.,   0., 929.,  44.]])
Labels (first 5 if any):
tensor([0])
Texts (first 5 if any):
['P

In [33]:
import os
import data.coru
# 1. Check where you are and if the data exists (helps prevent cryptic errors)
print("Current directory:", os.getcwd())
coru_path = '/content/CORU'
print(f"Does CORU path exist? {os.path.exists(coru_path)}")

# If you uploaded to Google Drive, your path might look like:
# coru_path = '/content/drive/MyDrive/data/CORU'
import json
from data.coru import build_handwriting_dataset_coru

# Build exactly 1 batch (since batch_size=16 in coru.py)
ds = build_handwriting_dataset_coru(
    dataset_name=coru_path,
    split='train',
    max_samples=64,  # 1 batch
    augment_prob=0.0, # Turn off aug to see the raw crop
)

print(f"Total rows in dataset: {len(ds)}")
print(f"Columns: {ds.column_names}\n")

# Look at the first row beautifully formatted
for i in range(18,23):
    row_0 = ds[i]
    print("--- Row 0 Structure ---")
    print(json.dumps(row_0, indent=2, ensure_ascii=False))

    # Verify the temp image was actually saved
    img_path = row_0['messages'][0]['content'][0]['image']
    print(f"\n--- Image Check ---")
    print(f"Temp image path: {img_path}")
    print(f"Does file exist? {os.path.exists(img_path)}")
    if os.path.exists(img_path):
        from PIL import Image
        img = Image.open(img_path)
        print(f"Image size (after crop): {img.size}")

Current directory: /content/drive/MyDrive/Project2_OCR
Does CORU path exist? True
[CORU] Loading from /content/CORU/OCR/train/train


CORU: augment + build messages:   0%|          | 0/64 [00:00<?, ? examples/s]

Total rows in dataset: 64
Columns: ['messages']

--- Row 0 Structure ---
{
  "messages": [
    {
      "content": [
        {
          "image": "/tmp/hajji_909990148.png",
          "text": null,
          "type": "image"
        },
        {
          "image": null,
          "text": "You are an expert Arabic handwriting recognition system.\nLook at the image and transcribe exactly what is written — nothing more, nothing less.\nRules:\n- If the image contains one short line, output only that line.\n- Keep Eastern Arabic digits (٠١٢٣٤٥٦٧٨٩) exactly as they appear.\n- Keep slashes, dots, and punctuation exactly as visible.\n- If the writing has no diacritics (tashkeel), do not invent diacritics.\n- Do not continue, explain, translate, or add any text beyond what is in the image.\n- Preserve original spelling.\nOutput only the transcribed Arabic text.",
          "type": "text"
        }
      ],
      "role": "user"
    },
    {
      "content": [
        {
          "image": null,
   

In [32]:
!python train.py \
    --use_coru \
    --coru_root /content/CORU \
    --coru_split train \
    --no_khatt \
    --output_dir ./outputs/coru_receipt_lora

CUDA: NOT available — training will use CPU (very slow). On WSL, install an NVIDIA driver that supports WSL2 and a GPU-enabled `pip install torch` wheel (see pytorch.org).
Fetching 2 files: 100% 2/2 [00:00<00:00, 4865.78it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 729/729 [00:00<00:00, 5979.01it/s]
Creating new LoRA adapter...
trainable params: 22,436,224 || all params: 2,231,421,824 || trainable%: 1.0055
Model staying on CPU: cpu
[Mixed] Including CORU: /content/CORU
[CORU] Loading local files from /content/CORU/OCR/train/train
CORU: augment + build messages: 100% 21103/21103 [02:40<00:00, 131.89 examples/s]
Mixed dataset total: 21103 rows (shuffled)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's valu